### Cell 04.01 — notebook setup and paths

In [ ]:
# Cell 04.01
# QTL mapping — setup

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats


def find_project_root():
    """Locate the repository root from the current working directory."""
    current = Path.cwd().resolve()
    for path in (current, *current.parents):
        if (path / "data").exists() and (path / "notebooks").exists():
            return path
    raise FileNotFoundError(
        "Repository root not found. Run this notebook from within "
        "the flyer-hartwig-qtl-reanalysis repository."
    )

PROJECT_ROOT = find_project_root()

GENOTYPE_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "fxh_genotypes.xlsx"
)

PHENOTYPE_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "fxh_phenotypes.xlsx"
)

MAP_FILE = (
    PROJECT_ROOT
    / "results"
    / "linkage_map"
    / "flyer_hartwig_structural_physical_map_final.xlsx"
)


print("QTL MAPPING NOTEBOOK")
print("=" * 100)

for label, path in [
    ("Genotypes", GENOTYPE_FILE),
    ("Phenotypes", PHENOTYPE_FILE),
    ("Structural map", MAP_FILE),
]:
    print(f"{label:20s}: {path.exists()}  {path}")

### Cell 04.02 — load the frozen QTL map

In [ ]:
# Cell 04.02
# Load frozen structural linkage map.

qtl_map = pd.read_excel(
    MAP_FILE,
    sheet_name="ordered_framework"
)


required_map_columns = [
    "structural_group",
    "structural_order",
    "marker",
    "kosambi_cm_structural_provisional",
    "dominant_chr",
    "assignment_status"
]


missing_columns = [
    col
    for col in required_map_columns
    if col not in qtl_map.columns
]

assert not missing_columns, (
    f"Missing required map columns: {missing_columns}"
)


qtl_map = (
    qtl_map
    .sort_values(
        ["structural_group", "structural_order"]
    )
    .reset_index(drop=True)
)


print("FROZEN QTL MAP LOADED")
print("=" * 100)

print("Markers:", len(qtl_map))

print(
    "Structural fragments:",
    qtl_map["structural_group"].nunique()
)

print(
    "Provisional Kosambi length:",
    round(
        qtl_map.groupby("structural_group")[
            "kosambi_cm_structural_provisional"
        ].max().sum(),
        3
    ),
    "cM"
)


assert len(qtl_map) == 166
assert qtl_map["structural_group"].nunique() == 22

print("\nMAP CHECKS PASSED.")

display(
    qtl_map[
        required_map_columns
    ].head(20)
)

### Cell 04.03 — load raw genotype and phenotype data

In [ ]:
# Cell 04.03
# Load raw genotype and phenotype datasets.

genotypes_raw = pd.read_excel(
    GENOTYPE_FILE,
    sheet_name="fxh_genotypes"
)

phenotypes_raw = pd.read_excel(
    PHENOTYPE_FILE,
    sheet_name="fxh_phenotypes"
)


print("RAW DATA LOADED")
print("=" * 100)

print(
    "Genotype matrix:",
    genotypes_raw.shape
)

print(
    "Phenotype matrix:",
    phenotypes_raw.shape
)

print(
    "\nGenotype ID column:",
    genotypes_raw.columns[0]
)

print(
    "Phenotype ID column:",
    phenotypes_raw.columns[0]
)


print("\nFirst genotype IDs:")
print(
    genotypes_raw.iloc[:5, 0].tolist()
)

print("\nFirst phenotype IDs:")
print(
    phenotypes_raw.iloc[:5, 0].tolist()
)

### Cell 04.04 — construct the conservative 92-RIL analysis population

In [ ]:
# Cell 04.04
# Define the conservative QTL analysis population.
#
# These two RILs were excluded during genotype QC because
# of >50% missing genotype calls.

EXCLUDED_RILS = [
    "fxh_ril_03",
    "fxh_ril_43"
]


genotype_id_col = genotypes_raw.columns[0]
phenotype_id_col = phenotypes_raw.columns[0]


# Normalize IDs defensively.
genotypes_raw[genotype_id_col] = (
    genotypes_raw[genotype_id_col]
    .astype(str)
    .str.strip()
)

phenotypes_raw[phenotype_id_col] = (
    phenotypes_raw[phenotype_id_col]
    .astype(str)
    .str.strip()
)


# Keep only RIL rows in phenotype table.
phenotype_rils = (
    phenotypes_raw.loc[
        phenotypes_raw[
            phenotype_id_col
        ].str.startswith(
            "fxh_ril_",
            na=False
        )
    ]
    .copy()
)


genotype_qtl = (
    genotypes_raw.loc[
        ~genotypes_raw[
            genotype_id_col
        ].isin(EXCLUDED_RILS)
    ]
    .copy()
)


phenotype_qtl = (
    phenotype_rils.loc[
        ~phenotype_rils[
            phenotype_id_col
        ].isin(EXCLUDED_RILS)
    ]
    .copy()
)


geno_ids = set(
    genotype_qtl[genotype_id_col]
)

pheno_ids = set(
    phenotype_qtl[phenotype_id_col]
)


print("QTL ANALYSIS POPULATION")
print("=" * 100)

print(
    "Genotype RILs after QC:",
    len(genotype_qtl)
)

print(
    "Phenotype RILs after QC:",
    len(phenotype_qtl)
)

print(
    "Shared RILs:",
    len(geno_ids & pheno_ids)
)

print(
    "\nGenotype-only IDs:",
    sorted(geno_ids - pheno_ids)
)

print(
    "Phenotype-only IDs:",
    sorted(pheno_ids - geno_ids)
)


assert len(genotype_qtl) == 92
assert len(geno_ids & pheno_ids) == 92

print("\n92-RIL POPULATION ALIGNMENT PASSED.")

### Cell 04.05 — extract the 166 framework markers

In [ ]:
# Cell 04.05
# Extract the exact 166 ordered framework markers from the raw genotype matrix.

framework_markers = (
    qtl_map["marker"]
    .astype(str)
    .tolist()
)


genotype_columns = set(
    genotypes_raw.columns.astype(str)
)


missing_framework_markers = [
    marker
    for marker in framework_markers
    if marker not in genotype_columns
]


print("FRAMEWORK MARKER EXTRACTION CHECK")
print("=" * 100)

print(
    "Framework markers expected:",
    len(framework_markers)
)

print(
    "Framework markers found in genotype matrix:",
    len(framework_markers) - len(missing_framework_markers)
)

print(
    "Missing framework markers:",
    missing_framework_markers
)


assert len(framework_markers) == 166

assert len(missing_framework_markers) == 0, (
    "Some frozen-map markers are absent from the raw genotype matrix."
)


# Keep ID + exactly the 166 ordered framework markers
genotype_framework = (
    genotype_qtl[
        [genotype_id_col] + framework_markers
    ]
    .copy()
)


print(
    "\nFramework genotype matrix shape:",
    genotype_framework.shape
)

assert genotype_framework.shape == (92, 167)

print("\nALL 166 FRAMEWORK MARKERS RECOVERED.")

### Cell 04.06 — normalize and verify genotype coding

In [ ]:
# Cell 04.06
# Normalize genotype calls and verify that the QTL matrix
# contains only 0, 2, and missing values.

geno_calls = (
    genotype_framework[
        framework_markers
    ]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
)


observed_calls = sorted(
    pd.unique(
        geno_calls.values.ravel()
    )
)


observed_nonmissing = [
    x
    for x in observed_calls
    if not pd.isna(x)
]


print("GENOTYPE CODING CHECK")
print("=" * 100)

print(
    "Observed non-missing genotype calls:",
    observed_nonmissing
)


unexpected_calls = [
    x
    for x in observed_nonmissing
    if x not in [0, 2]
]


print(
    "Unexpected genotype calls:",
    unexpected_calls
)


assert len(unexpected_calls) == 0


overall_missing = (
    geno_calls
    .isna()
    .mean()
    .mean()
)


print(
    f"\nOverall missingness across the 166-marker "
    f"QTL framework: {overall_missing:.3%}"
)


# Put normalized numeric calls back into analysis table
genotype_framework.loc[
    :,
    framework_markers
] = geno_calls


print("\nGENOTYPE CODING CHECK PASSED.")

### Cell 04.07 — align phenotype rows exactly to genotype rows

In [ ]:
# Cell 04.07
# Force phenotype order to match genotype order exactly.

ril_order = (
    genotype_framework[
        genotype_id_col
    ]
    .tolist()
)


phenotype_aligned = (
    phenotype_qtl
    .set_index(phenotype_id_col)
    .loc[ril_order]
    .reset_index()
)


print("RIL ORDER ALIGNMENT")
print("=" * 100)

print(
    "Genotype rows:",
    len(genotype_framework)
)

print(
    "Phenotype rows:",
    len(phenotype_aligned)
)


alignment_ok = (
    genotype_framework[
        genotype_id_col
    ].tolist()
    ==
    phenotype_aligned[
        phenotype_id_col
    ].tolist()
)


print(
    "Row order identical:",
    alignment_ok
)


assert alignment_ok

print("\nGENOTYPE–PHENOTYPE ROW ALIGNMENT PASSED.")

### Cell 04.08 — trait-level sample-size and missingness audit

In [ ]:
# Cell 04.08
# Summarize usable phenotype observations for all traits.

trait_columns = [
    col
    for col in phenotype_aligned.columns
    if col != phenotype_id_col
]


trait_qc = []

for trait in trait_columns:

    y = pd.to_numeric(
        phenotype_aligned[trait],
        errors="coerce"
    )

    n_nonmissing = y.notna().sum()
    n_missing = y.isna().sum()

    trait_qc.append({
        "trait": trait,
        "n_total_rils": len(y),
        "n_nonmissing": n_nonmissing,
        "n_missing": n_missing,
        "missing_fraction": n_missing / len(y),
        "mean": y.mean(),
        "sd": y.std(),
        "min": y.min(),
        "max": y.max()
    })


trait_qc = (
    pd.DataFrame(trait_qc)
    .sort_values(
        [
            "n_nonmissing",
            "trait"
        ],
        ascending=[
            False,
            True
        ]
    )
    .reset_index(drop=True)
)


print("TRAIT-LEVEL QTL SAMPLE SIZE AUDIT")
print("=" * 120)

print(
    "Traits:",
    len(trait_qc)
)

display(trait_qc)


print("\nTRAITS WITH MISSING PHENOTYPES")
print("-" * 120)

display(
    trait_qc.loc[
        trait_qc[
            "n_missing"
        ] > 0
    ]
)

### Cell 04.09 — marker-level QTL QC

In [ ]:
# Cell 04.09
# Marker-level sample size, missingness, and allele counts
# in the final 92-RIL / 166-marker QTL matrix.

marker_qc = []

for marker in framework_markers:

    g = pd.to_numeric(
        genotype_framework[marker],
        errors="coerce"
    )

    n0 = (g == 0).sum()
    n2 = (g == 2).sum()
    n_missing = g.isna().sum()
    n_nonmissing = g.notna().sum()

    marker_qc.append({
        "marker": marker,
        "n_nonmissing": n_nonmissing,
        "n_missing": n_missing,
        "missing_fraction": n_missing / len(g),
        "n_genotype_0": n0,
        "n_genotype_2": n2,
        "minor_genotype_count": min(n0, n2)
    })


marker_qc = pd.DataFrame(marker_qc)


# Attach map information
marker_qc = (
    qtl_map[
        [
            "structural_group",
            "structural_order",
            "marker",
            "kosambi_cm_structural_provisional",
            "dominant_chr",
            "assignment_status"
        ]
    ]
    .merge(
        marker_qc,
        on="marker",
        how="left",
        validate="one_to_one"
    )
)


print("QTL MARKER QC")
print("=" * 120)

print("Markers:", len(marker_qc))

print(
    "Median non-missing RILs:",
    marker_qc["n_nonmissing"].median()
)

print(
    "Minimum non-missing RILs:",
    marker_qc["n_nonmissing"].min()
)

print(
    "Maximum missing fraction:",
    f"{marker_qc['missing_fraction'].max():.1%}"
)

print(
    "Minimum minor-genotype count:",
    marker_qc["minor_genotype_count"].min()
)


display(
    marker_qc.sort_values(
        "missing_fraction",
        ascending=False
    ).head(15)
)

### Cell 04.10 — build the single-marker QTL scan function
* This uses ordinary least-squares regression:

$$ y = \beta_0+\beta_1g+\epsilon $$

* with genotype coded 0 versus 2.
* For now, the estimated effect is simply genotype-2 mean minus genotype-0 mean. We will not call that a Hartwig or Flyer effect until parental coding is confirmed.

In [ ]:
# Cell 04.10
# Single-marker QTL scan engine.
#
# Returns:
# - effective sample size
# - genotype-class means
# - allele-class difference
# - R²
# - nominal p-value
# - LOD score

def single_marker_qtl_scan(
    trait,
    genotype_df,
    phenotype_df,
    markers,
    map_df,
    min_n=20,
    min_genotype_class=5
):

    y_all = pd.to_numeric(
        phenotype_df[trait],
        errors="coerce"
    )

    results = []

    for marker in markers:

        g_all = pd.to_numeric(
            genotype_df[marker],
            errors="coerce"
        )

        valid = (
            y_all.notna()
            & g_all.notna()
        )

        y = y_all.loc[valid].astype(float)
        g = g_all.loc[valid].astype(float)

        n = len(y)

        if n < min_n:
            continue

        n0 = (g == 0).sum()
        n2 = (g == 2).sum()

        if min(n0, n2) < min_genotype_class:
            continue

        # Null model
        y_mean = y.mean()
        rss0 = np.sum(
            (y - y_mean) ** 2
        )

        # Regression model
        X = np.column_stack([
            np.ones(n),
            g.values
        ])

        beta, _, _, _ = np.linalg.lstsq(
            X,
            y.values,
            rcond=None
        )

        yhat = X @ beta

        rss1 = np.sum(
            (y.values - yhat) ** 2
        )

        # R²
        r2 = (
            1 - rss1 / rss0
            if rss0 > 0
            else np.nan
        )

        # LOD
        lod = (
            (n / 2.0)
            * np.log10(rss0 / rss1)
            if rss1 > 0 and rss0 > 0
            else np.nan
        )

        # Standard regression p-value
        slope, intercept, r_value, p_value, std_err = (
            stats.linregress(
                g.values,
                y.values
            )
        )

        mean0 = y.loc[g == 0].mean()
        mean2 = y.loc[g == 2].mean()

        results.append({
            "trait": trait,
            "marker": marker,
            "n": n,
            "n0": n0,
            "n2": n2,
            "mean_genotype_0": mean0,
            "mean_genotype_2": mean2,
            "effect_2_minus_0": mean2 - mean0,
            "r2": r2,
            "lod": lod,
            "p_nominal": p_value
        })


    results = pd.DataFrame(results)

    results = (
        results
        .merge(
            map_df[
                [
                    "marker",
                    "structural_group",
                    "structural_order",
                    "kosambi_cm_structural_provisional",
                    "dominant_chr",
                    "assignment_status"
                ]
            ],
            on="marker",
            how="left",
            validate="many_to_one"
        )
    )

    return (
        results
        .sort_values(
            "lod",
            ascending=False
        )
        .reset_index(drop=True)
    )


print("Single-marker QTL scan function ready.")

### Cell 04.11 — first diagnostic scans: SCN and SDS
* Let's first see whether the scan behaves biologically and statistically sensibly before doing permutations.

In [ ]:
# Cell 04.11
# Run initial scans for the major disease-resistance traits.

priority_traits = [
    "scn_fi3",
    "scn_fi14",
    "sds_di",
    "sds_ds",
    "sds_dx",
    "sds_dx_mean"
]


priority_scans = {}


for trait in priority_traits:

    scan = single_marker_qtl_scan(
        trait=trait,
        genotype_df=genotype_framework,
        phenotype_df=phenotype_aligned,
        markers=framework_markers,
        map_df=qtl_map,
        min_n=20,
        min_genotype_class=5
    )

    priority_scans[trait] = scan

    print("\n")
    print("=" * 120)
    print(f"TRAIT: {trait}")
    print("=" * 120)

    print(
        "Markers successfully tested:",
        len(scan)
    )

    display(
        scan[
            [
                "trait",
                "marker",
                "structural_group",
                "dominant_chr",
                "kosambi_cm_structural_provisional",
                "n",
                "mean_genotype_0",
                "mean_genotype_2",
                "effect_2_minus_0",
                "r2",
                "lod",
                "p_nominal"
            ]
        ]
        .head(10)
    )

### Cell 04.12 — summarize the strongest peak for each priority trait

In [ ]:
# Cell 04.12
# Extract the strongest observed single-marker peak
# for each priority disease trait.

priority_peak_summary = []

for trait, scan in priority_scans.items():

    if len(scan) == 0:
        continue

    peak = scan.iloc[0]

    priority_peak_summary.append({
        "trait": trait,
        "peak_marker": peak["marker"],
        "structural_group": peak["structural_group"],
        "candidate_chr": peak["dominant_chr"],
        "position_cm": peak[
            "kosambi_cm_structural_provisional"
        ],
        "n": peak["n"],
        "effect_2_minus_0": peak[
            "effect_2_minus_0"
        ],
        "r2": peak["r2"],
        "lod": peak["lod"],
        "p_nominal": peak["p_nominal"]
    })


priority_peak_summary = (
    pd.DataFrame(priority_peak_summary)
    .sort_values(
        "lod",
        ascending=False
    )
    .reset_index(drop=True)
)


print("TOP OBSERVED PEAKS — DISEASE TRAITS")
print("=" * 120)

display(priority_peak_summary)

### Cell 04.13 — permutation function

In [ ]:
# Cell 04.13 — corrected
# Genome-wide permutation test for maximum single-marker LOD.
#
# IMPORTANT:
# Phenotype missingness stays attached to the same RILs.
# Only OBSERVED phenotype values are permuted among phenotyped RILs.

def permutation_max_lod(
    trait,
    genotype_df,
    phenotype_df,
    markers,
    n_permutations=1000,
    min_n=20,
    min_genotype_class=5,
    random_seed=20260911
):

    rng = np.random.default_rng(random_seed)

    y_original = pd.to_numeric(
        phenotype_df[trait],
        errors="coerce"
    ).to_numpy(dtype=float)

    phenotype_observed = np.isfinite(
        y_original
    )

    observed_values = (
        y_original[
            phenotype_observed
        ]
        .copy()
    )

    print(
        f"{trait}: phenotyped RILs = "
        f"{len(observed_values)}"
    )

    # Pre-convert marker genotypes once.
    genotype_arrays = {
        marker: pd.to_numeric(
            genotype_df[marker],
            errors="coerce"
        ).to_numpy(dtype=float)
        for marker in markers
    }

    max_lods = np.full(
        n_permutations,
        np.nan,
        dtype=float
    )

    for perm in range(
        n_permutations
    ):

        # Preserve phenotype missingness positions.
        y_perm = np.full(
            len(y_original),
            np.nan,
            dtype=float
        )

        y_perm[
            phenotype_observed
        ] = rng.permutation(
            observed_values
        )

        permutation_lods = []

        for marker in markers:

            g = genotype_arrays[
                marker
            ]

            valid = (
                phenotype_observed
                & np.isfinite(g)
            )

            y = y_perm[
                valid
            ]

            x = g[
                valid
            ]

            n = len(y)

            if n < min_n:
                continue

            n0 = np.sum(
                x == 0
            )

            n2 = np.sum(
                x == 2
            )

            if min(
                n0,
                n2
            ) < min_genotype_class:
                continue

            rss0 = np.sum(
                (
                    y
                    - np.mean(y)
                ) ** 2
            )

            if rss0 <= 0:
                continue

            X = np.column_stack(
                [
                    np.ones(n),
                    x
                ]
            )

            beta, _, _, _ = (
                np.linalg.lstsq(
                    X,
                    y,
                    rcond=None
                )
            )

            residuals = (
                y
                - X @ beta
            )

            rss1 = np.sum(
                residuals ** 2
            )

            if rss1 <= 0:
                continue

            lod = (
                n / 2.0
            ) * np.log10(
                rss0 / rss1
            )

            permutation_lods.append(
                lod
            )

        if permutation_lods:

            max_lods[
                perm
            ] = np.max(
                permutation_lods
            )

        if (
            perm + 1
        ) % 100 == 0:

            print(
                f"{trait}: "
                f"{perm + 1}/"
                f"{n_permutations} "
                "permutations completed"
            )

    return max_lods


print(
    "Corrected permutation function ready."
)

### Cell 04.14 — run 1,000 permutations for the six disease traits

In [ ]:
# Cell 04.14
# Empirical genome-wide thresholds for SCN and SDS traits.

N_PERMUTATIONS = 1000

permutation_results = {}
permutation_thresholds = []


for i, trait in enumerate(priority_traits):

    print("\n")
    print("=" * 100)
    print(f"PERMUTATION TEST: {trait}")
    print("=" * 100)

    max_lods = permutation_max_lod(
        trait=trait,
        genotype_df=genotype_framework,
        phenotype_df=phenotype_aligned,
        markers=framework_markers,
        n_permutations=N_PERMUTATIONS,
        min_n=20,
        min_genotype_class=5,
        random_seed=20260911 + i
    )

    permutation_results[trait] = max_lods

    valid_lods = max_lods[
        np.isfinite(max_lods)
    ]

    threshold_10 = np.quantile(
        valid_lods,
        0.90
    )

    threshold_05 = np.quantile(
        valid_lods,
        0.95
    )

    threshold_01 = np.quantile(
        valid_lods,
        0.99
    )

    observed_peak = (
        priority_scans[trait]["lod"].max()
    )

    empirical_p = (
        1
        + np.sum(
            valid_lods >= observed_peak
        )
    ) / (
        len(valid_lods) + 1
    )

    permutation_thresholds.append({
        "trait": trait,
        "observed_peak_lod": observed_peak,
        "lod_threshold_10pct": threshold_10,
        "lod_threshold_05pct": threshold_05,
        "lod_threshold_01pct": threshold_01,
        "peak_empirical_p": empirical_p,
        "n_permutations": len(valid_lods)
    })


permutation_thresholds = (
    pd.DataFrame(
        permutation_thresholds
    )
    .sort_values(
        "observed_peak_lod",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\n")
print("EMPIRICAL GENOME-WIDE THRESHOLDS")
print("=" * 120)

display(permutation_thresholds)

### Cell 04.15 — classify the observed peaks

In [ ]:
# Cell 04.15
# Compare observed QTL peaks with empirical genome-wide thresholds.

disease_qtl_peaks = (
    priority_peak_summary
    .merge(
        permutation_thresholds[
            [
                "trait",
                "lod_threshold_10pct",
                "lod_threshold_05pct",
                "lod_threshold_01pct",
                "peak_empirical_p"
            ]
        ],
        on="trait",
        how="left",
        validate="one_to_one"
    )
)


def classify_qtl(row):

    if (
        row["lod"]
        >= row["lod_threshold_01pct"]
    ):
        return "significant_1pct"

    elif (
        row["lod"]
        >= row["lod_threshold_05pct"]
    ):
        return "significant_5pct"

    elif (
        row["lod"]
        >= row["lod_threshold_10pct"]
    ):
        return "suggestive_10pct"

    else:
        return "not_genomewide_significant"


disease_qtl_peaks[
    "genomewide_status"
] = disease_qtl_peaks.apply(
    classify_qtl,
    axis=1
)


print("DISEASE QTL PEAK CLASSIFICATION")
print("=" * 140)

display(
    disease_qtl_peaks[
        [
            "trait",
            "peak_marker",
            "structural_group",
            "candidate_chr",
            "position_cm",
            "n",
            "r2",
            "lod",
            "lod_threshold_10pct",
            "lod_threshold_05pct",
            "lod_threshold_01pct",
            "peak_empirical_p",
            "genomewide_status"
        ]
    ]
)

### Cell 04.16 — visualize permutation distributions

In [ ]:
# Cell 04.16 — revised
# Plot AND save permutation distributions for all priority disease traits.

FIGURE_DIR = (
    PROJECT_ROOT
    / "results"
    / "figures"
    / "qtl"
)

FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


saved_permutation_figures = []


for trait in priority_traits:

    max_lods = permutation_results[trait]
    max_lods = max_lods[
        np.isfinite(max_lods)
    ]

    observed = (
        priority_scans[trait]["lod"].max()
    )

    threshold_05 = (
        permutation_thresholds
        .loc[
            permutation_thresholds["trait"] == trait,
            "lod_threshold_05pct"
        ]
        .iloc[0]
    )

    fig, ax = plt.subplots(
        figsize=(8, 5)
    )

    ax.hist(
        max_lods,
        bins=30
    )

    ax.axvline(
        threshold_05,
        linestyle="--",
        linewidth=2,
        label=(
            f"5% threshold = "
            f"{threshold_05:.2f}"
        )
    )

    ax.axvline(
        observed,
        linestyle="-",
        linewidth=2,
        label=(
            f"Observed peak = "
            f"{observed:.2f}"
        )
    )

    ax.set_xlabel(
        "Maximum genome-wide LOD"
    )

    ax.set_ylabel(
        "Permutation count"
    )

    ax.set_title(
        f"Permutation test — {trait}"
    )

    ax.legend()

    fig.tight_layout()

    # Save high-resolution PNG
    output_file = (
        FIGURE_DIR
        / f"permutation_{trait}.png"
    )

    fig.savefig(
        output_file,
        dpi=300,
        bbox_inches="tight"
    )

    saved_permutation_figures.append(
        output_file
    )

    plt.show()

    print(
        f"Saved: {output_file}"
    )


print("\n")
print("PERMUTATION FIGURES SAVED")
print("=" * 100)

for path in saved_permutation_figures:
    print(path)

print(
    "\nNumber of figures saved:",
    len(saved_permutation_figures)
)

assert (
    len(saved_permutation_figures)
    == len(priority_traits)
)

assert all(
    path.exists()
    for path in saved_permutation_figures
)

print("\nFIGURE EXPORT CHECK PASSED.")

### Cell 04.17 — verify the permutation design
* This checks that observed marker sample sizes are preserved conceptually by the corrected procedure.

In [ ]:
# Cell 04.17
# Verify marker-specific sample sizes for the priority traits.
#
# These n-values should now be fixed across permutations because
# phenotype missingness remains attached to the original RILs.

permutation_n_audit = []


for trait in priority_traits:

    y = pd.to_numeric(
        phenotype_aligned[
            trait
        ],
        errors="coerce"
    )

    for marker in framework_markers:

        g = pd.to_numeric(
            genotype_framework[
                marker
            ],
            errors="coerce"
        )

        valid = (
            y.notna()
            & g.notna()
        )

        permutation_n_audit.append({
            "trait": trait,
            "marker": marker,
            "n_fixed": valid.sum()
        })


permutation_n_audit = pd.DataFrame(
    permutation_n_audit
)


permutation_n_summary = (
    permutation_n_audit
    .groupby(
        "trait"
    )["n_fixed"]
    .agg(
        [
            "min",
            "median",
            "max"
        ]
    )
    .reset_index()
)


print(
    "FIXED SAMPLE SIZE BY TRAIT "
    "ACROSS PERMUTATIONS"
)
print("=" * 100)

display(
    permutation_n_summary
)

### Cell 04.18 — prepare complete disease-trait scan table

In [ ]:
# Cell 04.18
# Combine all priority disease scans into one long table.

disease_scan_long = pd.concat(
    [
        priority_scans[trait].copy()
        for trait in priority_traits
    ],
    ignore_index=True
)


# Attach permutation thresholds
disease_scan_long = (
    disease_scan_long
    .merge(
        permutation_thresholds[
            [
                "trait",
                "lod_threshold_10pct",
                "lod_threshold_05pct"
            ]
        ],
        on="trait",
        how="left",
        validate="many_to_one"
    )
)


print("COMBINED DISEASE SCAN TABLE")
print("=" * 100)

print(
    "Rows:",
    len(disease_scan_long)
)

print(
    "Traits:",
    disease_scan_long[
        "trait"
    ].nunique()
)

print(
    "Markers per trait:"
)

display(
    disease_scan_long
    .groupby("trait")
    .size()
    .rename("n_markers")
    .reset_index()
)

### Cell 04.19 — linkage-fragment LOD profile plots
* This saves one figure per trait.

In [ ]:
# Cell 04.19
# Plot full LOD profiles by structural linkage fragment.

LOD_PROFILE_DIR = (
    PROJECT_ROOT
    / "results"
    / "figures"
    / "qtl"
    / "lod_profiles"
)

LOD_PROFILE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


saved_lod_profiles = []


for trait in priority_traits:

    scan = (
        priority_scans[trait]
        .sort_values(
            [
                "structural_group",
                "structural_order"
            ]
        )
        .copy()
    )

    threshold_10 = (
        permutation_thresholds
        .loc[
            permutation_thresholds[
                "trait"
            ] == trait,
            "lod_threshold_10pct"
        ]
        .iloc[0]
    )

    threshold_05 = (
        permutation_thresholds
        .loc[
            permutation_thresholds[
                "trait"
            ] == trait,
            "lod_threshold_05pct"
        ]
        .iloc[0]
    )


    groups = (
        scan[
            "structural_group"
        ]
        .drop_duplicates()
        .tolist()
    )


    # Build cumulative plotting coordinate
    offsets = {}
    current_offset = 0.0

    plot_rows = []

    group_centers = {}

    for group in groups:

        temp = (
            scan.loc[
                scan[
                    "structural_group"
                ] == group
            ]
            .sort_values(
                "kosambi_cm_structural_provisional"
            )
            .copy()
        )

        temp[
            "plot_x"
        ] = (
            temp[
                "kosambi_cm_structural_provisional"
            ]
            + current_offset
        )

        plot_rows.append(
            temp
        )

        group_min = (
            temp[
                "plot_x"
            ].min()
        )

        group_max = (
            temp[
                "plot_x"
            ].max()
        )

        group_centers[
            group
        ] = (
            group_min
            + group_max
        ) / 2.0

        current_offset = (
            group_max
            + 8.0
        )


    plot_df = pd.concat(
        plot_rows,
        ignore_index=True
    )


    fig, ax = plt.subplots(
        figsize=(16, 6)
    )


    for group in groups:

        temp = (
            plot_df.loc[
                plot_df[
                    "structural_group"
                ] == group
            ]
            .sort_values(
                "plot_x"
            )
        )

        ax.plot(
            temp[
                "plot_x"
            ],
            temp[
                "lod"
            ],
            marker="o",
            markersize=3,
            linewidth=1
        )


    ax.axhline(
        threshold_10,
        linestyle=":",
        linewidth=1.5,
        label=(
            f"10% genome-wide = "
            f"{threshold_10:.2f}"
        )
    )

    ax.axhline(
        threshold_05,
        linestyle="--",
        linewidth=1.5,
        label=(
            f"5% genome-wide = "
            f"{threshold_05:.2f}"
        )
    )


    ax.set_xticks(
        list(
            group_centers.values()
        )
    )

    ax.set_xticklabels(
        list(
            group_centers.keys()
        ),
        rotation=90
    )

    ax.set_xlabel(
        "Structural linkage fragment"
    )

    ax.set_ylabel(
        "LOD"
    )

    ax.set_title(
        f"Genome-wide single-marker LOD profile — {trait}"
    )

    ax.legend()

    fig.tight_layout()


    output_file = (
        LOD_PROFILE_DIR
        / f"lod_profile_{trait}.png"
    )

    fig.savefig(
        output_file,
        dpi=300,
        bbox_inches="tight"
    )

    saved_lod_profiles.append(
        output_file
    )

    plt.show()

    print(
        f"Saved: {output_file}"
    )


print(
    "\nLOD profile figures saved:",
    len(saved_lod_profiles)
)

### Cell 04.20 — identify local peak clusters
* This summarizes the strongest marker within each structural fragment.

In [ ]:
# Cell 04.20
# Extract the strongest marker per linkage fragment for each trait.

fragment_peak_rows = []


for trait in priority_traits:

    scan = (
        priority_scans[
            trait
        ]
        .copy()
    )

    fragment_peaks = (
        scan
        .sort_values(
            "lod",
            ascending=False
        )
        .groupby(
            "structural_group",
            as_index=False
        )
        .first()
    )


    fragment_peaks[
        "trait"
    ] = trait

    fragment_peak_rows.append(
        fragment_peaks
    )


fragment_peak_summary = (
    pd.concat(
        fragment_peak_rows,
        ignore_index=True
    )
    .sort_values(
        [
            "trait",
            "lod"
        ],
        ascending=[
            True,
            False
        ]
    )
)


print(
    "TOP FRAGMENT-SPECIFIC PEAKS"
)
print("=" * 140)


display(
    fragment_peak_summary[
        [
            "trait",
            "structural_group",
            "dominant_chr",
            "marker",
            "kosambi_cm_structural_provisional",
            "n",
            "r2",
            "lod",
            "p_nominal"
        ]
    ]
    .groupby(
        "trait",
        group_keys=False
    )
    .head(5)
)

### Cell 04.21 — inspect neighborhoods around the main SCN and SDS candidates

In [ ]:
# Cell 04.21
# Show local marker neighborhoods around the main candidate regions.

candidate_regions = [
    ("scn_fi3", "pLG01b_Gm20"),
    ("scn_fi14", "pLG01b_Gm20"),
    ("sds_ds", "pLG18"),
    ("sds_dx", "pLG18"),
    ("sds_dx_mean", "pLG18")
]


for trait, group in candidate_regions:

    print("\n")
    print("=" * 120)
    print(
        f"{trait} — {group}"
    )
    print("=" * 120)

    region = (
        priority_scans[
            trait
        ]
        .loc[
            priority_scans[
                trait
            ][
                "structural_group"
            ] == group
        ]
        .sort_values(
            "structural_order"
        )
    )

    display(
        region[
            [
                "marker",
                "structural_order",
                "kosambi_cm_structural_provisional",
                "n",
                "mean_genotype_0",
                "mean_genotype_2",
                "effect_2_minus_0",
                "r2",
                "lod",
                "p_nominal"
            ]
        ]
    )

### Cell 04.22 — annotate parental allele effects

In [ ]:
# Cell 04.22
# Translate genotype codes into parental allele effects.

GENOTYPE_PARENT = {
    0: "Hartwig",
    2: "Flyer"
}


disease_peak_allele_summary = (
    disease_qtl_peaks
    .copy()
)


# Recover genotype-class means from the full scans.
peak_mean_rows = []

for trait in priority_traits:

    peak_marker = (
        disease_peak_allele_summary
        .loc[
            disease_peak_allele_summary["trait"] == trait,
            "peak_marker"
        ]
        .iloc[0]
    )

    row = (
        priority_scans[trait]
        .loc[
            priority_scans[trait]["marker"] == peak_marker
        ]
        .iloc[0]
    )

    peak_mean_rows.append({
        "trait": trait,
        "peak_marker": peak_marker,
        "mean_hartwig_allele": row["mean_genotype_0"],
        "mean_flyer_allele": row["mean_genotype_2"],
        "flyer_minus_hartwig": row["effect_2_minus_0"]
    })


peak_mean_rows = pd.DataFrame(
    peak_mean_rows
)


disease_peak_allele_summary = (
    disease_peak_allele_summary
    .merge(
        peak_mean_rows,
        on=[
            "trait",
            "peak_marker"
        ],
        how="left",
        validate="one_to_one"
    )
)


def favorable_parent(row):

    # Disease traits:
    # lower value = greater resistance.

    if (
        row["mean_flyer_allele"]
        < row["mean_hartwig_allele"]
    ):
        return "Flyer"

    elif (
        row["mean_hartwig_allele"]
        < row["mean_flyer_allele"]
    ):
        return "Hartwig"

    else:
        return "equal"


disease_peak_allele_summary[
    "lower_disease_parent"
] = disease_peak_allele_summary.apply(
    favorable_parent,
    axis=1
)


print("PARENTAL EFFECTS AT DISEASE PEAKS")
print("=" * 150)

display(
    disease_peak_allele_summary[
        [
            "trait",
            "peak_marker",
            "structural_group",
            "candidate_chr",
            "n",
            "mean_hartwig_allele",
            "mean_flyer_allele",
            "flyer_minus_hartwig",
            "r2",
            "lod",
            "peak_empirical_p",
            "genomewide_status",
            "lower_disease_parent"
        ]
    ]
)

### Cell 04.23 — genotype-effect plots for the main SCN and SDS candidates

In [ ]:
# Cell 04.23
# Plot genotype-class effects at the principal disease candidate loci.

EFFECT_FIGURE_DIR = (
    PROJECT_ROOT
    / "results"
    / "figures"
    / "qtl"
    / "allele_effects"
)

EFFECT_FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


candidate_effects = [
    ("scn_fi3", "Satt354"),
    ("scn_fi14", "Satt270"),
    ("sds_ds", "Satt334"),
    ("sds_dx", "Satt334"),
    ("sds_dx_mean", "Satt334"),
    ("sds_di", "Satt488")
]


for trait, marker in candidate_effects:

    y = pd.to_numeric(
        phenotype_aligned[trait],
        errors="coerce"
    )

    g = pd.to_numeric(
        genotype_framework[marker],
        errors="coerce"
    )

    valid = (
        y.notna()
        & g.notna()
    )

    plot_df = pd.DataFrame({
        "phenotype": y.loc[valid],
        "parent": g.loc[valid].map(
            GENOTYPE_PARENT
        )
    })


    fig, ax = plt.subplots(
        figsize=(6, 5)
    )


    data = [
        plot_df.loc[
            plot_df["parent"] == "Hartwig",
            "phenotype"
        ].values,

        plot_df.loc[
            plot_df["parent"] == "Flyer",
            "phenotype"
        ].values
    ]


    ax.boxplot(
        data,
        tick_labels=[
            "Hartwig allele",
            "Flyer allele"
        ],
        showmeans=True
    )

    ax.set_ylabel(
        trait
    )

    ax.set_title(
        f"{trait} — {marker}"
    )

    fig.tight_layout()


    output_file = (
        EFFECT_FIGURE_DIR
        / f"allele_effect_{trait}_{marker}.png"
    )

    fig.savefig(
        output_file,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    print(
        f"Saved: {output_file}"
    )

### Cell 04.24 — 10,000-permutation confirmation for SCN FI3
* Because FI3 is close to the 5% threshold, this is one place where increasing permutation precision is worthwhile.

In [ ]:
# Cell 04.24
# High-resolution empirical significance test for the
# strongest disease signal: SCN FI3.

SCN_CONFIRM_PERMUTATIONS = 10000


scn_fi3_max_lods_10k = (
    permutation_max_lod(
        trait="scn_fi3",
        genotype_df=genotype_framework,
        phenotype_df=phenotype_aligned,
        markers=framework_markers,
        n_permutations=SCN_CONFIRM_PERMUTATIONS,
        min_n=20,
        min_genotype_class=5,
        random_seed=20260911
    )
)


valid_lods = (
    scn_fi3_max_lods_10k[
        np.isfinite(
            scn_fi3_max_lods_10k
        )
    ]
)


scn_fi3_observed = (
    priority_scans[
        "scn_fi3"
    ]["lod"].max()
)


scn_fi3_threshold_10 = np.quantile(
    valid_lods,
    0.90
)

scn_fi3_threshold_05 = np.quantile(
    valid_lods,
    0.95
)

scn_fi3_threshold_01 = np.quantile(
    valid_lods,
    0.99
)


scn_fi3_empirical_p = (
    1
    + np.sum(
        valid_lods >= scn_fi3_observed
    )
) / (
    len(valid_lods) + 1
)


print("SCN FI3 — 10,000 PERMUTATION CONFIRMATION")
print("=" * 100)

print(
    f"Observed peak LOD: "
    f"{scn_fi3_observed:.4f}"
)

print(
    f"10% genome-wide threshold: "
    f"{scn_fi3_threshold_10:.4f}"
)

print(
    f"5% genome-wide threshold: "
    f"{scn_fi3_threshold_05:.4f}"
)

print(
    f"1% genome-wide threshold: "
    f"{scn_fi3_threshold_01:.4f}"
)

print(
    f"Empirical genome-wide P: "
    f"{scn_fi3_empirical_p:.5f}"
)

### Cell 04.25 — save the high-resolution SCN permutation figure

In [ ]:
# Cell 04.25
# Save final high-resolution SCN FI3 permutation plot.

fig, ax = plt.subplots(
    figsize=(8, 5)
)


ax.hist(
    valid_lods,
    bins=40
)


ax.axvline(
    scn_fi3_threshold_10,
    linestyle=":",
    linewidth=2,
    label=(
        f"10% threshold = "
        f"{scn_fi3_threshold_10:.2f}"
    )
)


ax.axvline(
    scn_fi3_threshold_05,
    linestyle="--",
    linewidth=2,
    label=(
        f"5% threshold = "
        f"{scn_fi3_threshold_05:.2f}"
    )
)


ax.axvline(
    scn_fi3_observed,
    linestyle="-",
    linewidth=2,
    label=(
        f"Observed = "
        f"{scn_fi3_observed:.2f}"
    )
)


ax.set_xlabel(
    "Maximum genome-wide LOD"
)

ax.set_ylabel(
    "Permutation count"
)

ax.set_title(
    "SCN FI3 — 10,000-permutation genome-wide test"
)

ax.legend()

fig.tight_layout()


SCN_CONFIRM_FIGURE = (
    PROJECT_ROOT
    / "results"
    / "figures"
    / "qtl"
    / "scn_fi3_10000_permutation_test.png"
)


fig.savefig(
    SCN_CONFIRM_FIGURE,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


print(
    f"Saved: {SCN_CONFIRM_FIGURE}"
)

### Cell 04.26 — run all 33 traits

In [ ]:
# Cell 04.26
# Run the baseline single-marker scan for all 33 phenotypes.

all_trait_scans = {}

for trait in trait_columns:

    scan = single_marker_qtl_scan(
        trait=trait,
        genotype_df=genotype_framework,
        phenotype_df=phenotype_aligned,
        markers=framework_markers,
        map_df=qtl_map,
        min_n=20,
        min_genotype_class=5
    )

    all_trait_scans[trait] = scan

    print(
        f"{trait:20s} "
        f"markers tested = {len(scan):3d}, "
        f"peak LOD = {scan['lod'].max():.3f}"
    )

### Cell 04.27 — summarize top peak per trait

In [ ]:
# Cell 04.27
# Extract strongest marker for every phenotype.

all_trait_peak_rows = []

for trait, scan in all_trait_scans.items():

    if len(scan) == 0:
        continue

    peak = scan.iloc[0]

    all_trait_peak_rows.append({
        "trait": trait,
        "peak_marker": peak["marker"],
        "structural_group": peak["structural_group"],
        "candidate_chr": peak["dominant_chr"],
        "position_cm": peak[
            "kosambi_cm_structural_provisional"
        ],
        "n": peak["n"],
        "mean_genotype_0": peak[
            "mean_genotype_0"
        ],
        "mean_genotype_2": peak[
            "mean_genotype_2"
        ],
        "flyer_minus_hartwig": peak[
            "effect_2_minus_0"
        ],
        "r2": peak["r2"],
        "lod": peak["lod"],
        "p_nominal": peak["p_nominal"]
    })


all_trait_peak_summary = (
    pd.DataFrame(
        all_trait_peak_rows
    )
    .sort_values(
        "lod",
        ascending=False
    )
    .reset_index(drop=True)
)


print("TOP PEAK FOR ALL 33 TRAITS")
print("=" * 140)

display(
    all_trait_peak_summary
)

### Cell 04.28 — identify traits worth permutation testing
* Rather than running 1,000 permutations blindly for all 33 traits, first rank the observed peaks.

In [ ]:
# Cell 04.28
# Rank traits by observed peak strength.

all_trait_peak_summary[
    "lower_value_parent"
] = np.where(
    all_trait_peak_summary[
        "mean_genotype_2"
    ]
    <
    all_trait_peak_summary[
        "mean_genotype_0"
    ],
    "Flyer",
    "Hartwig"
)


print("TRAITS RANKED BY PEAK LOD")
print("=" * 140)

display(
    all_trait_peak_summary[
        [
            "trait",
            "peak_marker",
            "structural_group",
            "candidate_chr",
            "n",
            "r2",
            "lod",
            "p_nominal",
            "flyer_minus_hartwig",
            "lower_value_parent"
        ]
    ]
)

### Cell 04.29 — flag strongest candidates

In [ ]:
# Cell 04.29
# Flag the strongest observed genome-wide peaks for follow-up.

high_priority_traits = (
    all_trait_peak_summary
    .loc[
        all_trait_peak_summary[
            "lod"
        ] >= 2.0,
        "trait"
    ]
    .tolist()
)


moderate_priority_traits = (
    all_trait_peak_summary
    .loc[
        (
            all_trait_peak_summary[
                "lod"
            ] >= 1.5
        )
        &
        (
            all_trait_peak_summary[
                "lod"
            ] < 2.0
        ),
        "trait"
    ]
    .tolist()
)


print("HIGH-PRIORITY TRAITS")
print("=" * 100)
print(high_priority_traits)

print("\nMODERATE-PRIORITY TRAITS")
print("=" * 100)
print(moderate_priority_traits)

print(
    "\nNumber high priority:",
    len(high_priority_traits)
)

print(
    "Number moderate priority:",
    len(moderate_priority_traits)
)

### Cell 04.30 — define the non-disease permutation set

In [ ]:
# Cell 04.30
# Select high-priority NON-DISEASE traits for empirical permutation testing.

non_disease_priority_traits = [
    "prot_03",
    "days_fl_07",
    "oil_2003",
    "brt",
    "seed_wt",
    "lrn",
    "oil_2001",
    "prot_05"
]


print("NON-DISEASE TRAITS FOR PERMUTATION TESTING")
print("=" * 100)

for trait in non_disease_priority_traits:

    peak = (
        all_trait_peak_summary
        .loc[
            all_trait_peak_summary["trait"] == trait
        ]
        .iloc[0]
    )

    print(
        f"{trait:15s}  "
        f"LOD={peak['lod']:.3f}  "
        f"marker={peak['peak_marker']:12s}  "
        f"group={peak['structural_group']:15s}  "
        f"chr={peak['candidate_chr']}  "
        f"n={int(peak['n'])}"
    )


assert len(non_disease_priority_traits) == 8

print("\nSelection confirmed.")

### Cell 04.31 — run 1,000 permutations for those 8 traits
* This uses the corrected permutation function that keeps phenotype missingness attached to the original RILs.

In [ ]:
# Cell 04.31
# Trait-specific 1,000-permutation genome-wide tests
# for the 8 strongest non-disease traits.

NON_DISEASE_N_PERM = 1000

non_disease_permutation_results = {}
non_disease_threshold_rows = []


for i, trait in enumerate(
    non_disease_priority_traits
):

    print("\n")
    print("=" * 100)
    print(
        f"PERMUTATION TEST: {trait}"
    )
    print("=" * 100)

    max_lods = permutation_max_lod(
        trait=trait,
        genotype_df=genotype_framework,
        phenotype_df=phenotype_aligned,
        markers=framework_markers,
        n_permutations=NON_DISEASE_N_PERM,
        min_n=20,
        min_genotype_class=5,
        random_seed=20261000 + i
    )

    non_disease_permutation_results[
        trait
    ] = max_lods

    valid_lods = max_lods[
        np.isfinite(max_lods)
    ]

    observed_peak = (
        all_trait_scans[
            trait
        ]["lod"].max()
    )

    threshold_10 = np.quantile(
        valid_lods,
        0.90
    )

    threshold_05 = np.quantile(
        valid_lods,
        0.95
    )

    threshold_01 = np.quantile(
        valid_lods,
        0.99
    )

    empirical_p = (
        1
        + np.sum(
            valid_lods >= observed_peak
        )
    ) / (
        len(valid_lods) + 1
    )

    non_disease_threshold_rows.append({
        "trait": trait,
        "observed_peak_lod": observed_peak,
        "lod_threshold_10pct": threshold_10,
        "lod_threshold_05pct": threshold_05,
        "lod_threshold_01pct": threshold_01,
        "peak_empirical_p": empirical_p,
        "n_permutations": len(valid_lods)
    })


non_disease_thresholds = (
    pd.DataFrame(
        non_disease_threshold_rows
    )
    .sort_values(
        "observed_peak_lod",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\n")
print(
    "NON-DISEASE EMPIRICAL GENOME-WIDE THRESHOLDS"
)
print("=" * 130)

display(
    non_disease_thresholds
)

### Cell 04.32 — classify those peaks

In [ ]:
# Cell 04.32
# Classify high-priority non-disease peaks
# against empirical genome-wide thresholds.

non_disease_peak_classification = (
    all_trait_peak_summary
    .loc[
        all_trait_peak_summary[
            "trait"
        ].isin(
            non_disease_priority_traits
        )
    ]
    .merge(
        non_disease_thresholds[
            [
                "trait",
                "lod_threshold_10pct",
                "lod_threshold_05pct",
                "lod_threshold_01pct",
                "peak_empirical_p"
            ]
        ],
        on="trait",
        how="left",
        validate="one_to_one"
    )
)


def classify_genomewide_peak(row):

    if (
        row["lod"]
        >= row["lod_threshold_01pct"]
    ):
        return "significant_1pct"

    elif (
        row["lod"]
        >= row["lod_threshold_05pct"]
    ):
        return "significant_5pct"

    elif (
        row["lod"]
        >= row["lod_threshold_10pct"]
    ):
        return "suggestive_10pct"

    else:
        return "not_genomewide_significant"


non_disease_peak_classification[
    "genomewide_status"
] = (
    non_disease_peak_classification
    .apply(
        classify_genomewide_peak,
        axis=1
    )
)


print(
    "NON-DISEASE PEAK CLASSIFICATION"
)
print("=" * 150)

display(
    non_disease_peak_classification[
        [
            "trait",
            "peak_marker",
            "structural_group",
            "candidate_chr",
            "n",
            "mean_genotype_0",
            "mean_genotype_2",
            "flyer_minus_hartwig",
            "r2",
            "lod",
            "lod_threshold_10pct",
            "lod_threshold_05pct",
            "peak_empirical_p",
            "genomewide_status"
        ]
    ]
    .sort_values(
        "lod",
        ascending=False
    )
)

### Cell 04.33 — combined summary of all permutation-tested traits
* This gives us one master table containing the disease results plus these new non-disease results.

In [ ]:
# Cell 04.33
# Combine disease and non-disease empirical results
# into a single master QTL-screen summary.

# Use the corrected 1,000-permutation disease table,
# but replace SCN FI3 with its more precise 10,000-permutation result.

disease_empirical_final = (
    disease_qtl_peaks
    .copy()
)


# Update SCN FI3 with 10k result
mask = (
    disease_empirical_final[
        "trait"
    ] == "scn_fi3"
)

disease_empirical_final.loc[
    mask,
    "lod_threshold_10pct"
] = scn_fi3_threshold_10

disease_empirical_final.loc[
    mask,
    "lod_threshold_05pct"
] = scn_fi3_threshold_05

disease_empirical_final.loc[
    mask,
    "lod_threshold_01pct"
] = scn_fi3_threshold_01

disease_empirical_final.loc[
    mask,
    "peak_empirical_p"
] = scn_fi3_empirical_p


# Reclassify after 10k update
disease_empirical_final[
    "genomewide_status"
] = (
    disease_empirical_final
    .apply(
        classify_genomewide_peak,
        axis=1
    )
)


# Standardize columns
disease_master = (
    disease_empirical_final[
        [
            "trait",
            "peak_marker",
            "structural_group",
            "candidate_chr",
            "n",
            "r2",
            "lod",
            "lod_threshold_10pct",
            "lod_threshold_05pct",
            "lod_threshold_01pct",
            "peak_empirical_p",
            "genomewide_status"
        ]
    ]
    .copy()
)


non_disease_master = (
    non_disease_peak_classification[
        [
            "trait",
            "peak_marker",
            "structural_group",
            "candidate_chr",
            "n",
            "r2",
            "lod",
            "lod_threshold_10pct",
            "lod_threshold_05pct",
            "lod_threshold_01pct",
            "peak_empirical_p",
            "genomewide_status"
        ]
    ]
    .copy()
)


permutation_tested_trait_summary = (
    pd.concat(
        [
            disease_master,
            non_disease_master
        ],
        ignore_index=True
    )
    .sort_values(
        [
            "peak_empirical_p",
            "lod"
        ],
        ascending=[
            True,
            False
        ]
    )
    .reset_index(drop=True)
)


print(
    "MASTER EMPIRICAL QTL SCREEN"
)
print("=" * 150)

display(
    permutation_tested_trait_summary
)


print("\nSTATUS COUNTS")
print("-" * 100)

display(
    permutation_tested_trait_summary[
        "genomewide_status"
    ]
    .value_counts()
    .rename_axis(
        "status"
    )
    .reset_index(
        name="n_traits"
    )
)

### Cell 04.34 — 10,000-permutation confirmation of the three non-disease suggestive signals

In [ ]:
# Cell 04.34
# High-resolution confirmation of the three
# suggestive non-disease QTL signals.

confirmation_traits = [
    "lrn",
    "prot_03",
    "days_fl_07"
]

N_CONFIRM_PERM = 10000

confirmation_permutation_results = {}
confirmation_rows = []


for i, trait in enumerate(
    confirmation_traits
):

    print("\n")
    print("=" * 100)
    print(
        f"10,000-PERMUTATION CONFIRMATION: {trait}"
    )
    print("=" * 100)

    max_lods = permutation_max_lod(
        trait=trait,
        genotype_df=genotype_framework,
        phenotype_df=phenotype_aligned,
        markers=framework_markers,
        n_permutations=N_CONFIRM_PERM,
        min_n=20,
        min_genotype_class=5,
        random_seed=20262000 + i
    )

    confirmation_permutation_results[
        trait
    ] = max_lods

    valid_lods = max_lods[
        np.isfinite(max_lods)
    ]

    observed_lod = (
        all_trait_scans[
            trait
        ]["lod"].max()
    )

    threshold_10 = np.quantile(
        valid_lods,
        0.90
    )

    threshold_05 = np.quantile(
        valid_lods,
        0.95
    )

    threshold_01 = np.quantile(
        valid_lods,
        0.99
    )

    empirical_p = (
        1
        + np.sum(
            valid_lods >= observed_lod
        )
    ) / (
        len(valid_lods) + 1
    )

    confirmation_rows.append({
        "trait": trait,
        "observed_lod": observed_lod,
        "lod_threshold_10pct": threshold_10,
        "lod_threshold_05pct": threshold_05,
        "lod_threshold_01pct": threshold_01,
        "empirical_p": empirical_p,
        "n_permutations": len(valid_lods)
    })


confirmation_thresholds_10k = (
    pd.DataFrame(
        confirmation_rows
    )
    .sort_values(
        "empirical_p"
    )
    .reset_index(drop=True)
)


print("\n")
print(
    "10,000-PERMUTATION CONFIRMATION RESULTS"
)
print("=" * 130)

display(
    confirmation_thresholds_10k
)

### Cell 04.35 — classify the high-resolution results

In [ ]:
# Cell 04.35
# Final high-resolution classification.

def classify_confirmed_peak(row):

    if (
        row["observed_lod"]
        >= row["lod_threshold_01pct"]
    ):
        return "significant_1pct"

    elif (
        row["observed_lod"]
        >= row["lod_threshold_05pct"]
    ):
        return "significant_5pct"

    elif (
        row["observed_lod"]
        >= row["lod_threshold_10pct"]
    ):
        return "suggestive_10pct"

    else:
        return "not_genomewide_significant"


confirmation_thresholds_10k[
    "genomewide_status"
] = (
    confirmation_thresholds_10k
    .apply(
        classify_confirmed_peak,
        axis=1
    )
)


display(
    confirmation_thresholds_10k[
        [
            "trait",
            "observed_lod",
            "lod_threshold_10pct",
            "lod_threshold_05pct",
            "lod_threshold_01pct",
            "empirical_p",
            "genomewide_status"
        ]
    ]
)

### Cell 04.36 — update the master empirical table
* This replaces the 1,000-permutation estimates for these three traits with the 10,000-permutation estimates.

In [ ]:
# Cell 04.36
# Update master QTL table with high-resolution results.

qtl_empirical_final = (
    permutation_tested_trait_summary
    .copy()
)


for _, row in (
    confirmation_thresholds_10k
    .iterrows()
):

    trait = row["trait"]

    mask = (
        qtl_empirical_final[
            "trait"
        ] == trait
    )

    qtl_empirical_final.loc[
        mask,
        "lod_threshold_10pct"
    ] = row["lod_threshold_10pct"]

    qtl_empirical_final.loc[
        mask,
        "lod_threshold_05pct"
    ] = row["lod_threshold_05pct"]

    qtl_empirical_final.loc[
        mask,
        "lod_threshold_01pct"
    ] = row["lod_threshold_01pct"]

    qtl_empirical_final.loc[
        mask,
        "peak_empirical_p"
    ] = row["empirical_p"]

    qtl_empirical_final.loc[
        mask,
        "genomewide_status"
    ] = row["genomewide_status"]


qtl_empirical_final = (
    qtl_empirical_final
    .sort_values(
        [
            "peak_empirical_p",
            "lod"
        ],
        ascending=[
            True,
            False
        ]
    )
    .reset_index(drop=True)
)


print(
    "FINAL HIGH-PRIORITY EMPIRICAL QTL SUMMARY"
)
print("=" * 150)

display(
    qtl_empirical_final
)

### Cell 04.37 — save the current QTL checkpoint

In [ ]:
# Cell 04.37
# Save reproducible QTL checkpoint.

QTL_RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "qtl"
)

QTL_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


QTL_CHECKPOINT = (
    QTL_RESULTS_DIR
    / "flyer_hartwig_qtl_screen_checkpoint.xlsx"
)


with pd.ExcelWriter(
    QTL_CHECKPOINT,
    engine="openpyxl"
) as writer:

    all_trait_peak_summary.to_excel(
        writer,
        sheet_name="all_33_trait_peaks",
        index=False
    )

    qtl_empirical_final.to_excel(
        writer,
        sheet_name="empirical_tested_traits",
        index=False
    )

    disease_peak_allele_summary.to_excel(
        writer,
        sheet_name="disease_allele_effects",
        index=False
    )

    confirmation_thresholds_10k.to_excel(
        writer,
        sheet_name="non_disease_10k",
        index=False
    )


print(
    f"Saved QTL checkpoint:\n{QTL_CHECKPOINT}"
)

### Cell 04.38 — identify the traits that still need permutation testing

In [ ]:
# Cell 04.38
# Identify traits not yet empirically permutation-tested.

already_tested_traits = set(
    qtl_empirical_final["trait"]
)

all_traits_set = set(
    trait_columns
)

remaining_traits = [
    trait
    for trait in trait_columns
    if trait not in already_tested_traits
]


print("ALL TRAITS:", len(trait_columns))
print("ALREADY TESTED:", len(already_tested_traits))
print("REMAINING:", len(remaining_traits))

print("\nREMAINING TRAITS")
print("=" * 100)

for trait in remaining_traits:
    peak = (
        all_trait_peak_summary
        .loc[
            all_trait_peak_summary["trait"] == trait
        ]
        .iloc[0]
    )

    print(
        f"{trait:20s} "
        f"LOD={peak['lod']:.3f} "
        f"marker={peak['peak_marker']}"
    )

### Cell 04.39 — run 1,000 permutations for the remaining traits

In [ ]:
# Cell 04.39
# Complete the empirical genome-wide screen for all
# previously untested traits.

REMAINING_N_PERM = 1000

remaining_permutation_results = {}
remaining_threshold_rows = []


for i, trait in enumerate(
    remaining_traits
):

    print("\n")
    print("=" * 100)
    print(
        f"PERMUTATION TEST: {trait}"
    )
    print("=" * 100)

    max_lods = permutation_max_lod(
        trait=trait,
        genotype_df=genotype_framework,
        phenotype_df=phenotype_aligned,
        markers=framework_markers,
        n_permutations=REMAINING_N_PERM,
        min_n=20,
        min_genotype_class=5,
        random_seed=20263000 + i
    )

    remaining_permutation_results[
        trait
    ] = max_lods

    valid_lods = max_lods[
        np.isfinite(max_lods)
    ]

    observed_lod = (
        all_trait_scans[
            trait
        ]["lod"].max()
    )

    threshold_10 = np.quantile(
        valid_lods,
        0.90
    )

    threshold_05 = np.quantile(
        valid_lods,
        0.95
    )

    threshold_01 = np.quantile(
        valid_lods,
        0.99
    )

    empirical_p = (
        1
        + np.sum(
            valid_lods >= observed_lod
        )
    ) / (
        len(valid_lods) + 1
    )

    remaining_threshold_rows.append({
        "trait": trait,
        "observed_lod": observed_lod,
        "lod_threshold_10pct": threshold_10,
        "lod_threshold_05pct": threshold_05,
        "lod_threshold_01pct": threshold_01,
        "peak_empirical_p": empirical_p,
        "n_permutations": len(valid_lods)
    })


remaining_thresholds = (
    pd.DataFrame(
        remaining_threshold_rows
    )
    .sort_values(
        "peak_empirical_p"
    )
    .reset_index(drop=True)
)


display(
    remaining_thresholds
)

### Cell 04.40 — classify the remaining traits

In [ ]:
# Cell 04.40
# Classify all remaining trait peaks.

remaining_classification = (
    all_trait_peak_summary
    .loc[
        all_trait_peak_summary[
            "trait"
        ].isin(
            remaining_traits
        )
    ]
    .merge(
        remaining_thresholds[
            [
                "trait",
                "lod_threshold_10pct",
                "lod_threshold_05pct",
                "lod_threshold_01pct",
                "peak_empirical_p"
            ]
        ],
        on="trait",
        how="left",
        validate="one_to_one"
    )
)


remaining_classification[
    "genomewide_status"
] = (
    remaining_classification
    .apply(
        classify_genomewide_peak,
        axis=1
    )
)


print(
    "REMAINING TRAITS — EMPIRICAL CLASSIFICATION"
)
print("=" * 150)

display(
    remaining_classification[
        [
            "trait",
            "peak_marker",
            "structural_group",
            "candidate_chr",
            "n",
            "r2",
            "lod",
            "lod_threshold_10pct",
            "lod_threshold_05pct",
            "peak_empirical_p",
            "genomewide_status"
        ]
    ]
    .sort_values(
        "peak_empirical_p"
    )
)

### Cell 04.41 — build the true all-33-trait empirical summary

In [ ]:
# Cell 04.41
# Combine previously tested and newly tested traits
# into a complete all-33-trait empirical QTL screen.

previous_empirical = (
    qtl_empirical_final
    .copy()
)


# Fix any blank status values first.
previous_empirical[
    "genomewide_status"
] = previous_empirical.apply(
    classify_genomewide_peak,
    axis=1
)


remaining_master = (
    remaining_classification[
        [
            "trait",
            "peak_marker",
            "structural_group",
            "candidate_chr",
            "n",
            "r2",
            "lod",
            "lod_threshold_10pct",
            "lod_threshold_05pct",
            "lod_threshold_01pct",
            "peak_empirical_p",
            "genomewide_status"
        ]
    ]
    .copy()
)


all_33_empirical_summary = (
    pd.concat(
        [
            previous_empirical,
            remaining_master
        ],
        ignore_index=True
    )
    .drop_duplicates(
        subset="trait",
        keep="first"
    )
    .sort_values(
        [
            "peak_empirical_p",
            "lod"
        ],
        ascending=[
            True,
            False
        ]
    )
    .reset_index(drop=True)
)


assert (
    all_33_empirical_summary[
        "trait"
    ].nunique()
    == 33
)


print(
    "COMPLETE 33-TRAIT EMPIRICAL QTL SCREEN"
)
print("=" * 160)

display(
    all_33_empirical_summary
)


print("\nSTATUS COUNTS")
print("=" * 100)

display(
    all_33_empirical_summary[
        "genomewide_status"
    ]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="n_traits")
)

### Cell 04.42 — define the four suggestive regions

In [ ]:
# Cell 04.42
# Freeze the empirically suggestive traits from the complete 33-trait screen.

suggestive_traits = (
    all_33_empirical_summary
    .loc[
        all_33_empirical_summary[
            "genomewide_status"
        ] == "suggestive_10pct",
        "trait"
    ]
    .tolist()
)


print("SUGGESTIVE TRAITS")
print("=" * 100)

for trait in suggestive_traits:

    row = (
        all_33_empirical_summary
        .loc[
            all_33_empirical_summary[
                "trait"
            ] == trait
        ]
        .iloc[0]
    )

    print(
        f"{trait:15s} "
        f"{row['peak_marker']:12s} "
        f"{row['structural_group']:15s} "
        f"{row['candidate_chr']:6s} "
        f"LOD={row['lod']:.3f} "
        f"P={row['peak_empirical_p']:.4f}"
    )


assert len(suggestive_traits) == 4

### Cell 04.43 — inspect complete local marker neighborhoods

In [ ]:
# Cell 04.43
# Display every marker in the peak structural fragment
# for each suggestive trait.

suggestive_region_tables = {}


for trait in suggestive_traits:

    summary_row = (
        all_33_empirical_summary
        .loc[
            all_33_empirical_summary[
                "trait"
            ] == trait
        ]
        .iloc[0]
    )

    peak_group = summary_row[
        "structural_group"
    ]

    region = (
        all_trait_scans[
            trait
        ]
        .loc[
            all_trait_scans[
                trait
            ][
                "structural_group"
            ] == peak_group
        ]
        .sort_values(
            "structural_order"
        )
        .copy()
    )

    suggestive_region_tables[
        trait
    ] = region


    print("\n")
    print("=" * 130)
    print(
        f"{trait} — "
        f"{peak_group} — "
        f"{summary_row['candidate_chr']}"
    )
    print("=" * 130)

    display(
        region[
            [
                "marker",
                "structural_order",
                "kosambi_cm_structural_provisional",
                "n",
                "mean_genotype_0",
                "mean_genotype_2",
                "effect_2_minus_0",
                "r2",
                "lod",
                "p_nominal"
            ]
        ]
    )

### Cell 04.44 — descriptive 1-LOD regions
* Because this is a single-marker scan, these should not be called formal confidence intervals. They are descriptive local support regions only.

In [ ]:
# Cell 04.44
# Construct contiguous descriptive 1-LOD support regions
# around each suggestive peak.
#
# IMPORTANT:
# These are NOT formal QTL confidence intervals.
# They summarize neighboring markers within 1 LOD
# of the single-marker peak.

support_region_rows = []


for trait in suggestive_traits:

    scan = all_trait_scans[
        trait
    ].copy()

    peak_idx = (
        scan["lod"].idxmax()
    )

    peak_row = (
        scan.loc[
            peak_idx
        ]
    )

    peak_group = peak_row[
        "structural_group"
    ]

    group_scan = (
        scan.loc[
            scan[
                "structural_group"
            ] == peak_group
        ]
        .sort_values(
            "structural_order"
        )
        .reset_index(drop=True)
    )

    peak_position = (
        group_scan[
            "lod"
        ].idxmax()
    )

    peak_lod = (
        group_scan.loc[
            peak_position,
            "lod"
        ]
    )

    cutoff = peak_lod - 1.0


    # Expand left and right while LOD stays
    # at or above peak - 1.
    left = peak_position

    while (
        left > 0
        and group_scan.loc[
            left - 1,
            "lod"
        ] >= cutoff
    ):
        left -= 1


    right = peak_position

    while (
        right
        < len(group_scan) - 1
        and group_scan.loc[
            right + 1,
            "lod"
        ] >= cutoff
    ):
        right += 1


    support = (
        group_scan.iloc[
            left:right + 1
        ]
        .copy()
    )


    support_region_rows.append({
        "trait": trait,
        "peak_marker": peak_row["marker"],
        "structural_group": peak_group,
        "candidate_chr": peak_row["dominant_chr"],
        "peak_lod": peak_lod,
        "lod_minus_1_cutoff": cutoff,
        "left_marker": support.iloc[0]["marker"],
        "right_marker": support.iloc[-1]["marker"],
        "left_cm": support.iloc[0][
            "kosambi_cm_structural_provisional"
        ],
        "right_cm": support.iloc[-1][
            "kosambi_cm_structural_provisional"
        ],
        "n_markers_in_region": len(support)
    })


suggestive_support_regions = (
    pd.DataFrame(
        support_region_rows
    )
)


suggestive_support_regions[
    "span_cm"
] = (
    suggestive_support_regions[
        "right_cm"
    ]
    -
    suggestive_support_regions[
        "left_cm"
    ]
)


print(
    "DESCRIPTIVE 1-LOD REGIONS"
)
print("=" * 150)

display(
    suggestive_support_regions
)

### Cell 04.45 — final all-trait checkpoint

In [ ]:
# Cell 04.45
# Save the complete empirical QTL screen and
# suggestive-region summaries.

FINAL_QTL_SCREEN = (
    PROJECT_ROOT
    / "results"
    / "qtl"
    / "flyer_hartwig_all33_empirical_qtl_screen.xlsx"
)


with pd.ExcelWriter(
    FINAL_QTL_SCREEN,
    engine="openpyxl"
) as writer:

    all_33_empirical_summary.to_excel(
        writer,
        sheet_name="all_33_empirical",
        index=False
    )

    suggestive_support_regions.to_excel(
        writer,
        sheet_name="suggestive_regions",
        index=False
    )

    all_trait_peak_summary.to_excel(
        writer,
        sheet_name="all_trait_peaks",
        index=False
    )

    disease_peak_allele_summary.to_excel(
        writer,
        sheet_name="disease_allele_effects",
        index=False
    )

    for trait in suggestive_traits:

        suggestive_region_tables[
            trait
        ].to_excel(
            writer,
            sheet_name=f"region_{trait}"[:31],
            index=False
        )


print(
    "Saved final empirical QTL screen:"
)

print(
    FINAL_QTL_SCREEN
)

### Cell 04.46 — quantify regional support around each suggestive peak

In [ ]:
# Cell 04.46
# Quantify local regional support for the four suggestive loci.

regional_support_rows = []


for trait in suggestive_traits:

    region = (
        suggestive_region_tables[
            trait
        ]
        .sort_values(
            "structural_order"
        )
        .reset_index(drop=True)
        .copy()
    )

    peak_i = int(
        region["lod"].idxmax()
    )

    peak = region.loc[
        peak_i
    ]

    peak_effect = (
        peak["effect_2_minus_0"]
    )

    peak_sign = np.sign(
        peak_effect
    )

    # Immediate neighbors where available
    neighbor_idx = [
        i
        for i in [
            peak_i - 1,
            peak_i + 1
        ]
        if 0 <= i < len(region)
    ]

    neighbors = (
        region.loc[
            neighbor_idx
        ]
        .copy()
    )

    same_direction_neighbors = (
        np.sign(
            neighbors[
                "effect_2_minus_0"
            ]
        )
        == peak_sign
    ).sum()

    # All other markers in the fragment
    other = (
        region.drop(
            index=peak_i
        )
    )

    same_direction_all = (
        np.sign(
            other[
                "effect_2_minus_0"
            ]
        )
        == peak_sign
    ).sum()

    second_best_lod = (
        other["lod"].max()
        if len(other) > 0
        else np.nan
    )

    peak_at_edge = (
        peak_i == 0
        or
        peak_i == len(region) - 1
    )

    n_within_05_lod = (
        region[
            "lod"
        ]
        >= (
            peak["lod"] - 0.5
        )
    ).sum()

    n_within_10_lod = (
        region[
            "lod"
        ]
        >= (
            peak["lod"] - 1.0
        )
    ).sum()

    regional_support_rows.append({
        "trait": trait,
        "peak_marker": peak["marker"],
        "structural_group": peak[
            "structural_group"
        ],
        "candidate_chr": peak[
            "dominant_chr"
        ],
        "peak_lod": peak["lod"],
        "peak_effect_flyer_minus_hartwig":
            peak_effect,
        "peak_at_fragment_edge":
            peak_at_edge,
        "n_markers_in_fragment":
            len(region),
        "n_immediate_neighbors":
            len(neighbors),
        "same_direction_immediate_neighbors":
            same_direction_neighbors,
        "same_direction_other_markers":
            same_direction_all,
        "n_other_markers":
            len(other),
        "second_best_lod":
            second_best_lod,
        "lod_drop_to_second_best":
            (
                peak["lod"]
                - second_best_lod
                if np.isfinite(
                    second_best_lod
                )
                else np.nan
            ),
        "n_markers_within_0.5_lod":
            n_within_05_lod,
        "n_markers_within_1.0_lod":
            n_within_10_lod
    })


regional_support_summary = (
    pd.DataFrame(
        regional_support_rows
    )
)


display(
    regional_support_summary
)

### Cell 04.47 — assign descriptive regional-support categories

In [ ]:
# Cell 04.47
# Assign transparent descriptive categories.
# These are interpretation labels, NOT statistical tests.

def regional_pattern(row):

    if (
        row["n_markers_within_1.0_lod"] >= 2
        and not row["peak_at_fragment_edge"]
    ):
        return "multi_marker_supported"

    if (
        row["n_markers_within_1.0_lod"] >= 2
        and row["peak_at_fragment_edge"]
    ):
        return "multi_marker_edge_supported"

    if row["peak_at_fragment_edge"]:
        return "single_marker_edge_peak"

    if (
        row["same_direction_immediate_neighbors"]
        ==
        row["n_immediate_neighbors"]
        and row[
            "n_immediate_neighbors"
        ] > 0
    ):
        return "single_marker_peak_directionally_supported"

    return "single_marker_peak"


regional_support_summary[
    "regional_pattern"
] = (
    regional_support_summary
    .apply(
        regional_pattern,
        axis=1
    )
)


print(
    "REGIONAL SUPPORT CLASSIFICATION"
)
print("=" * 140)

display(
    regional_support_summary[
        [
            "trait",
            "peak_marker",
            "candidate_chr",
            "peak_lod",
            "peak_at_fragment_edge",
            "second_best_lod",
            "lod_drop_to_second_best",
            "n_markers_within_1.0_lod",
            "same_direction_immediate_neighbors",
            "regional_pattern"
        ]
    ]
)

### Cell 04.48 — make publication-style local LOD profile plots

In [ ]:
# Cell 04.48
# Plot local LOD profiles for the four suggestive regions.

import matplotlib.pyplot as plt


REGIONAL_FIG_DIR = (
    PROJECT_ROOT
    / "results"
    / "figures"
    / "qtl"
    / "suggestive_regions"
)

REGIONAL_FIG_DIR.mkdir(
    parents=True,
    exist_ok=True
)


for trait in suggestive_traits:

    region = (
        suggestive_region_tables[
            trait
        ]
        .sort_values(
            "kosambi_cm_structural_provisional"
        )
        .copy()
    )

    summary = (
        all_33_empirical_summary
        .loc[
            all_33_empirical_summary[
                "trait"
            ] == trait
        ]
        .iloc[0]
    )

    x = region[
        "kosambi_cm_structural_provisional"
    ]

    y = region[
        "lod"
    ]


    fig, ax = plt.subplots(
        figsize=(8, 5)
    )

    ax.plot(
        x,
        y,
        marker="o"
    )

    ax.axhline(
        summary[
            "lod_threshold_10pct"
        ],
        linestyle="--",
        label="10% genome-wide threshold"
    )

    ax.axhline(
        summary[
            "lod_threshold_05pct"
        ],
        linestyle=":",
        label="5% genome-wide threshold"
    )


    for _, row in region.iterrows():

        ax.annotate(
            row["marker"],
            (
                row[
                    "kosambi_cm_structural_provisional"
                ],
                row["lod"]
            ),
            xytext=(0, 6),
            textcoords="offset points",
            ha="center",
            fontsize=8,
            rotation=45
        )


    ax.set_xlabel(
        "Provisional genetic position (cM)"
    )

    ax.set_ylabel(
        "LOD score"
    )

    ax.set_title(
        f"{trait}: "
        f"{summary['structural_group']} "
        f"({summary['candidate_chr']})"
    )

    ax.legend()

    fig.tight_layout()


    out_file = (
        REGIONAL_FIG_DIR
        / f"{trait}_local_lod_profile.png"
    )

    fig.savefig(
        out_file,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    print(
        f"Saved: {out_file}"
    )

### Cell 04.49 — save the regional-evidence checkpoint

In [ ]:
# Cell 04.49
# Add regional-support evidence to the final QTL workbook.

REGIONAL_CHECKPOINT = (
    PROJECT_ROOT
    / "results"
    / "qtl"
    / "flyer_hartwig_suggestive_qtl_regions.xlsx"
)


with pd.ExcelWriter(
    REGIONAL_CHECKPOINT,
    engine="openpyxl"
) as writer:

    regional_support_summary.to_excel(
        writer,
        sheet_name="regional_support",
        index=False
    )

    suggestive_support_regions.to_excel(
        writer,
        sheet_name="descriptive_1LOD",
        index=False
    )

    for trait in suggestive_traits:

        suggestive_region_tables[
            trait
        ].to_excel(
            writer,
            sheet_name=f"region_{trait}"[:31],
            index=False
        )


print(
    f"Saved:\n{REGIONAL_CHECKPOINT}"
)